In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import numpy as np
import torch
import gpytorch
from tqdm.auto import tqdm
from linear_operator import settings
from gpytorch import settings as gsettings
from grf_gp.sampler import GRFSampler
from grf_gp.utils.spectral import get_normalized_laplacian
from grf_gp.kernels.diffusion import DiffusionGRFKernel
from grf_gp.kernels.general import GeneralGRFKernel
from grf_gp.model import GRFGP
from grf_gp.utils.config import set_gp_defaults

project_root = os.path.abspath('../../..')
sys.path.append(project_root)
sys.path.append(os.path.abspath('.'))
from data_utils import prepare_wind_graph_data
from experiments.regression.utils import train_model, evaluate_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_gp_defaults(settings, gsettings)

In [3]:
NC_FILE = os.path.join(project_root, 'data/wind_interpolation/aac1241c2ceff963a460829f1c68b132.nc')
USE_DOWNSAMPLING = True
DOWNSAMPLE_FACTOR = 20
MAX_WALK_LENGTH = 4
WALKS_PER_NODE = 2048
P_HALT = 0.05
LR = 0.05
MAX_ITER = 100
SEED = 0

In [4]:
np.random.seed(SEED)
torch.manual_seed(SEED)

data = prepare_wind_graph_data(
    NC_FILE,
    use_downsampling=USE_DOWNSAMPLING,
    downsample_factor=DOWNSAMPLE_FACTOR,
    dtype=np.float64,
)

X_train = torch.tensor(data['X_train']).long().to(device)
Y_train = torch.tensor(data['y_train'], dtype=torch.float64).flatten().to(device)
X_test = torch.tensor(data['X_test']).long().to(device)
Y_test = torch.tensor(data['y_test'], dtype=torch.float64).flatten().to(device)
X_full = torch.tensor(data['X']).long().to(device)

adjacency_matrix = data['adjacency'].toarray()
L = get_normalized_laplacian(adjacency_matrix)
L = torch.tensor(L, dtype=torch.float64).to(device)
orig_std = float(data['y_std'])

In [5]:
sampler = GRFSampler(L, WALKS_PER_NODE, P_HALT, MAX_WALK_LENGTH, seed=SEED, use_tqdm=True, n_processes=4)
rw_mats = sampler()

/Users/matthew/Documents/Efficient Gaussian Process on Graphs/Efficient_Gaussian_Process_On_Graphs/venv/lib/python3.11/site-packages/grf_gp/utils/csr.py:17: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:55.)
  return adjacency.to_sparse_csr()


In [6]:
likelihood_general = gpytorch.likelihoods.GaussianLikelihood().to(device)
kernel_general = GeneralGRFKernel(rw_mats, MAX_WALK_LENGTH).to(device)
model_general = GRFGP(X_train, Y_train, likelihood_general, kernel_general).to(device)
_ = train_model(model_general, likelihood_general, X_train, Y_train, lr=LR, max_iter=MAX_ITER)
lml_g, rmse_g, nlpd_g = evaluate_model(model_general, likelihood_general, X_train, Y_train, X_test, Y_test, orig_std)
print(f'GeneralGRFKernel | LML: {lml_g:.4f} | RMSE: {rmse_g:.4f} | NLPD: {nlpd_g:.4f}')
print('modulation:', kernel_general.modulation_function.detach().cpu().numpy())

Training:   0%|          | 0/100 [00:00<?, ?it/s]

/Users/matthew/Documents/Efficient Gaussian Process on Graphs/Efficient_Gaussian_Process_On_Graphs/venv/lib/python3.11/site-packages/linear_operator/utils/sparse.py:51: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  if nonzero_indices.storage():
/Users/matthew/Documents/Efficient Gaussian Process on Graphs/Efficient_Gaussian_Process_On_Graphs/venv/lib/python3.11/site-packages/linear_operator/utils/sparse.py:66: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:656.)
  res = cls(index_tensor, value_tensor, interp_size)
/Users/matthew/Doc

GeneralGRFKernel | LML: -441.0987 | RMSE: 2.2021 | NLPD: 1.3778
modulation: [ 1.9032434 -0.9994174 -1.1656892  0.5880569]


In [7]:
likelihood_diff = gpytorch.likelihoods.GaussianLikelihood().to(device)
kernel_diff = DiffusionGRFKernel(rw_mats, MAX_WALK_LENGTH).to(device)
model_diff = GRFGP(X_train, Y_train, likelihood_diff, kernel_diff).to(device)
_ = train_model(model_diff, likelihood_diff, X_train, Y_train, lr=LR, max_iter=MAX_ITER)
lml_d, rmse_d, nlpd_d = evaluate_model(model_diff, likelihood_diff, X_train, Y_train, X_test, Y_test, orig_std)
print(f'DiffusionGRFKernel | LML: {lml_d:.4f} | RMSE: {rmse_d:.4f} | NLPD: {nlpd_d:.4f}')
print(f'beta: {kernel_diff.beta.item():.4f}, sigma_f: {kernel_diff.sigma_f.item():.4f}')

Training:   0%|          | 0/100 [00:00<?, ?it/s]

DiffusionGRFKernel | LML: 68.9070 | RMSE: 2.1985 | NLPD: 1.3407
beta: 1.8335, sigma_f: 1.8200
